# Axral Vocal AI — Kaggle One-cell pipeline
**入力は `HF_DATA_URL` だけ**。Hub revision固定→UTAU関連ファイルだけを容量制限付きで取得→ZIP/UAR/TARを安全展開→文字コードに配慮した音源監査→pytest→承認済みラベルmanifestがある場合の学習・評価→成果物ZIPまで1セルで実行します。
> デフォルト上限はダウンロード4096 MiB・展開8192 MiBです。学習対象は現時点の実験用メタデータ・ピッチ差分モデルです。UTAU WAVだけから歌唱表現の正解ラベルは生成できないため、承認済みラベル付きJSONLが無ければ学習を安全に停止します。音声生成モデルの学習ではありません。

In [ ]:
# ============================================================# AXRAL VOCAL AI — ONE CELL KAGGLE PIPELINE# 設定は HF_DATA_URL だけ。取得→文字コード対策→監査→pytest→条件付き学習・評価→ZIP# ============================================================import os, re, sys, json, subprocess, traceback, zipfile, shutilfrom pathlib import Path# ★ここだけ変更：Hugging Face の Dataset URL（例 https://huggingface.co/datasets/OWNER/REPO）HF_DATA_URL = "https://huggingface.co/datasets/SEKOIA2900/VO-US"PROJECT = Path("/kaggle/working/Axral_vocal_AI")DATA_ROOT = Path("/kaggle/working/hf_voicebanks")OUT = Path("/kaggle/working/axral_run")if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)  # prevent stale files from previous dataset/revisionif OUT.exists(): shutil.rmtree(OUT)  # prevent stale reports from previous runsOUT.mkdir(parents=True, exist_ok=True)LOG = OUT / "pipeline.log"def log(msg):    print(msg)    with LOG.open("a", encoding="utf-8") as f: f.write(str(msg) + "\n")def run(cmd, check=True, output_tail=1200):    log("$ " + " ".join(map(str, cmd)))    p = subprocess.run(cmd, cwd=PROJECT, text=True, stdout=subprocess.PIPE,                       stderr=subprocess.STDOUT, errors="replace")    with LOG.open("a", encoding="utf-8") as f: f.write(p.stdout)    if output_tail and p.stdout:        print(p.stdout[-output_tail:])    else:        log(f"command exit={p.returncode}; full output saved to pipeline.log")    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")    return p.returncodeLOG.write_text("", encoding="utf-8")try:    assert HF_DATA_URL.startswith("https://huggingface.co/"), "HF_DATA_URL must be a Hugging Face URL"    required_pipeline_files = (        "scripts/kaggle_hf_download.py",        "scripts/voicebank_audit_summary.py",        "training/validate_dataset.py",        "training/train_pitch_model.py",        "src/axral_vocal_ai/voicebank_audit_cli.py",    )    if not PROJECT.exists():        log("Project checkout missing; cloning Axral_vocal_AI default branch.")        clone = subprocess.run(            ["git", "clone", "--depth", "1", "https://github.com/nebula-INE/Axral_vocal_AI.git", str(PROJECT)],            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, errors="replace"        )        log(clone.stdout)        if clone.returncode:            raise RuntimeError(f"Git clone failed ({clone.returncode})")    else:        if not (PROJECT / ".git").is_dir():            raise RuntimeError("Project path exists but is not an Axral_vocal_AI Git checkout; move it before running.")        status = subprocess.run(            ["git", "status", "--porcelain"],            cwd=PROJECT, text=True, stdout=subprocess.PIPE,            stderr=subprocess.STDOUT, errors="replace"        )        if status.returncode:            raise RuntimeError("Could not inspect existing Axral_vocal_AI checkout.")        if status.stdout.strip():            raise RuntimeError("Existing Axral_vocal_AI checkout has local changes; preserve them and use a clean Kaggle working directory.")        log("Refreshing existing Axral_vocal_AI checkout from origin/main.")        fetch = subprocess.run(            ["git", "fetch", "--depth", "1", "origin", "main"],            cwd=PROJECT, text=True, stdout=subprocess.PIPE,            stderr=subprocess.STDOUT, errors="replace"        )        log(fetch.stdout)        if fetch.returncode:            raise RuntimeError("Could not refresh Axral_vocal_AI from origin/main.")        reset = subprocess.run(            ["git", "reset", "--hard", "FETCH_HEAD"],            cwd=PROJECT, text=True, stdout=subprocess.PIPE,            stderr=subprocess.STDOUT, errors="replace"        )        log(reset.stdout)        if reset.returncode:            raise RuntimeError("Could not switch Axral_vocal_AI checkout to the fetched main revision.")    # PR-only notebook users must be able to run before this pipeline is merged.    # Prefer main after merge; use the feature branch only while required helpers are absent.    missing_pipeline_files = [        name for name in required_pipeline_files if not (PROJECT / name).is_file()    ]    if missing_pipeline_files:        log(            "Default branch does not contain the Kaggle pipeline helpers yet; "            "trying feat/kaggle-one-url-pipeline."        )        fetch = subprocess.run(            ["git", "fetch", "--depth", "1", "origin", "feat/kaggle-one-url-pipeline"],            cwd=PROJECT, text=True, stdout=subprocess.PIPE,            stderr=subprocess.STDOUT, errors="replace"        )        log(fetch.stdout)        if fetch.returncode:            raise RuntimeError("Could not fetch the Kaggle pipeline feature branch.")        reset = subprocess.run(            ["git", "reset", "--hard", "FETCH_HEAD"],            cwd=PROJECT, text=True, stdout=subprocess.PIPE,            stderr=subprocess.STDOUT, errors="replace"        )        log(reset.stdout)        if reset.returncode:            raise RuntimeError("Could not switch to the Kaggle pipeline feature branch.")        missing_pipeline_files = [            name for name in required_pipeline_files if not (PROJECT / name).is_file()        ]    if missing_pipeline_files:        raise RuntimeError(            "Axral checkout is missing required Kaggle pipeline files: "            + ", ".join(missing_pipeline_files)        )    source_commit = subprocess.run(        ["git", "rev-parse", "HEAD"],        cwd=PROJECT, text=True, stdout=subprocess.PIPE,        stderr=subprocess.STDOUT, errors="replace"    )    if source_commit.returncode:        raise RuntimeError("Could not record Axral_vocal_AI source revision.")    source_revision = source_commit.stdout.strip()    log(f"Axral_vocal_AI source commit: {source_revision}")    # The notebook kernel may start outside the repository directory.    if str(PROJECT) not in sys.path:        sys.path.insert(0, str(PROJECT))    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,training]"], output_tail=800)    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], output_tail=800)    # Resolve HF_TOKEN from the environment or Kaggle Secrets. Never log the token.    token = os.environ.get("HF_TOKEN")    if not token:        try:            from kaggle_secrets import UserSecretsClient            token = UserSecretsClient().get_secret("HF_TOKEN")        except Exception:            token = None  # Public datasets do not need a token.    if token:        os.environ["HF_TOKEN"] = token    # Download only voicebank archives/metadata/WAVs and JSONL manifests.    # The helper pins the Hub revision, enforces size limits, and safely extracts ZIP/tar archives.    log("[1/5] Resolve and download Hugging Face dataset")    run([        sys.executable, "scripts/kaggle_hf_download.py", HF_DATA_URL,        "--output-dir", str(DATA_ROOT),    ], output_tail=1800)    download_summary = json.loads(        (DATA_ROOT / "download_summary.json").read_text(encoding="utf-8")    )    (OUT / "download_summary.json").write_text(        json.dumps(download_summary, ensure_ascii=False, indent=2) + "\n",        encoding="utf-8",    )    log(        "Dataset card license declaration: "        + str(download_summary.get("dataset_card_license") or "not declared")        + f"; license/readme documents indexed: {len(download_summary.get('license_documents', []))}"    )    dataset_id = download_summary["dataset"]    # Encoding safety: audit text is decoded by project reader (BOM/UTF-16/UTF-8/CP932).    # Keep original voicebank files untouched; no destructive re-encoding.    log("[2/5] Locate voicebank roots")    from scripts.kaggle_hf_download import discover_voicebank_roots    banks = discover_voicebank_roots(DATA_ROOT / "downloads", DATA_ROOT / "extracted")    log(f"oto.ini roots: {len(banks)}")    if not banks: raise RuntimeError("No oto.ini found. Check HF dataset layout/download.")    log("[3/5] Voicebank audit (JSON per bank)")    audit_dir = OUT / "audits"; audit_dir.mkdir(exist_ok=True)    audit_results = []    for i, bank in enumerate(banks, 1):        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", bank.relative_to(DATA_ROOT).as_posix())[:100] or f"bank_{i}"        target = audit_dir / f"{i:03d}_{safe}.json"        audit_rc = run([sys.executable, "-m", "axral_vocal_ai.voicebank_audit_cli",                        str(bank), "--output", str(target)], check=False, output_tail=0)        audit_results.append((i, target, audit_rc))        log(f"audit {i}/{len(banks)}: bank_index={i}, exit={audit_rc}")    from scripts.voicebank_audit_summary import build_audit_index    audit_index = build_audit_index(audit_results)    audit_failures = audit_index["failures"]    (OUT / "voicebank_audit_summary.json").write_text(        json.dumps(audit_index, ensure_ascii=False, indent=2) + "\n",        encoding="utf-8",    )    log(        "Audit totals: "        + json.dumps(audit_index["aggregate"], ensure_ascii=False)        + f"; failed banks: {len(audit_failures)}"    )    log("[4/5] Tests")    test_rc = run([sys.executable, "-m", "pytest", "-q"], check=False, output_tail=2500)    log(f"pytest exit={test_rc}")    log("[5/5] Discover approved labelled training manifests")    manifests = sorted(        p for p in DATA_ROOT.rglob("*")        if p.is_file() and p.suffix.casefold() == ".jsonl"    )    log(f"JSONL candidates: {len(manifests)}")    eligible = []    validation_failures = []    training_status = "blocked_no_approved_labelled_manifest"    model_report = OUT / "models" / "pitch_delta_report.json"    if download_summary.get("archive_errors"):        training_status = "blocked_archive_extraction_errors"        log("Training blocked because one or more archives could not be safely extracted.")    elif audit_failures:        training_status = "blocked_voicebank_audit_findings"        log(f"Training blocked because {len(audit_failures)} voicebank audit(s) returned findings.")    elif test_rc != 0:        training_status = "blocked_tests_failed"        log("Training blocked because pytest failed.")    else:        # Validate every candidate using the repository's strict permission,        # label, split-isolation, path-redaction, and numeric-range checks.        for manifest in manifests:            rc = run([                sys.executable, "training/validate_dataset.py",                str(manifest), "--require-labeled"            ], check=False, output_tail=1000)            if rc == 0:                eligible.append(manifest)            else:                validation_failures.append({                    "manifest": manifest.relative_to(DATA_ROOT).as_posix(),                    "exit_code": rc                })        if len(eligible) == 1:            log(f"Using validated manifest: {eligible[0].relative_to(DATA_ROOT)}")            model_report.parent.mkdir(parents=True, exist_ok=True)            # This is the existing metadata-only ridge MVP, not a waveform trainer.            # Fixed alpha=1.0; evaluate the isolated test voicebank once at the end.            run([                sys.executable, "training/train_pitch_model.py",                str(eligible[0]), "--output", str(model_report), "--evaluate-test"            ], output_tail=2500)            training_status = "metadata_pitch_model_trained_and_tested"        elif len(eligible) > 1:            training_status = "blocked_multiple_eligible_manifests"            log("Training blocked: multiple eligible manifests found; refusing to choose one implicitly.")        else:            log("No eligible labelled JSONL manifest found.")            log("UTAU oto.ini/WAV inventory does not contain aligned score-to-pitch-delta labels.")            log("No model training was attempted; provide a reviewed pitch_training_manifest.jsonl in the HF dataset.")    summary = {        "dataset": dataset_id,        "source_commit": source_revision,        "hub_revision": download_summary.get("resolved_revision"),        "download": download_summary,        "voicebank_roots": len(banks),        "audit_files": len(list(audit_dir.glob("*.json"))),        "audit_failures": audit_failures,        "voicebank_audit_summary": str(OUT / "voicebank_audit_summary.json"),        "pytest_exit": test_rc,        "jsonl_candidates": len(manifests),        "eligible_manifests": len(eligible),        "manifest_validation_failures": validation_failures,        "training": training_status,        "model_report": str(model_report) if model_report.exists() else None,        "training_scope": "metadata_only_pitch_delta_ridge_mvp",        "waveform_model_training": "not_implemented",        "gpu_training_used": False    }    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")    log(json.dumps(summary, ensure_ascii=False))except Exception:    log(traceback.format_exc())    raisefinally:    log(f"LOG: {LOG}")    log(f"OUTPUT: {OUT}")    archive = OUT.parent / "axral_run_artifacts.zip"    log(f"ARTIFACT_ZIP: {archive}")    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:        for artifact in OUT.rglob("*"):            if artifact.is_file():                bundle.write(artifact, artifact.relative_to(OUT.parent))    try:        from IPython.display import FileLink, display        display(FileLink(str(archive)))    except Exception as display_error:        log(f"Artifact link display unavailable: {display_error}")